# telephony-codec-bench: Colab (SNAC vs EnCodec)

Telephony-degraded speech round-trip benchmark: **SNAC** vs **EnCodec**, STOI / PESQ / latency.

**GitHub:** [ST-48-1240162/telephony-codec-bench](https://github.com/ST-48-1240162/telephony-codec-bench). Push repo before first Colab run.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ST-48-1240162/telephony-codec-bench/blob/main/docs/Telephony_Codec_Bench.ipynb)

| Step | Time (T4) | Output |
|------|-----------|--------|
| noisekit data | 45-90 min @ 200 utt | `./data/telephony_speech` |
| full benchmark | ~30-60 min | `reports/benchmark.json` |

## 0. Runtime

**Runtime → Change runtime type → T4 GPU** (then run the next cell).

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("torch:", torch.__version__)
else:
    raise RuntimeError(
        "cuda: False — still on CPU runtime.\n"
        "1) Runtime → Change runtime type → T4 GPU\n"
        "2) Runtime → Restart session\n"
        "3) Rerun this cell until cuda: True\n"
        "Do NOT run install (§2) until GPU is available."
    )

## 1. Clone repo

Clone from GitHub (`ST-48-1240162`).  
Repo not pushed yet? **File → Upload notebook** to Colab, zip-upload the project to Drive, and `%cd` into it instead of cloning.

In [ ]:
GITHUB_USER = "ST-48-1240162"
REPO = f"https://github.com/{GITHUB_USER}/telephony-codec-bench.git"
BRANCH = "main"

!rm -rf telephony-codec-bench
!git clone --depth 1 --branch {BRANCH} {REPO}
%cd telephony-codec-bench

## 2. Install dependencies

In [ ]:
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU. Colab: Runtime → Change runtime type → T4 GPU, "
        "then Runtime → Restart session, rerun §0 and this cell."
    )

# 1) System libs
!apt-get -qq install -y libsndfile1 ffmpeg

# 3) Project deps first (pinned: docs/colab-requirements.txt)
!{sys.executable} -m pip install -q -r docs/colab-requirements.txt

# 4) Re-pin matched torch stack (cu128 trio; cu130 may lack matching torchaudio)
!{sys.executable} -m pip install -q --force-reinstall torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install -q -e . --no-deps
!{sys.executable} -m pip install -q "fsspec==2025.3.0" --force-reinstall --no-deps

# 5) Verify manifest (skip pip check: Colab base image has unrelated conflicts)
!{sys.executable} scripts/verify_colab_env.py --no-pip-check

## 3. Generate telephony data (noisekit)

Default dataset: `google/fleurs` (no extra license step).  
Quick try: `SAMPLES = 50`. Full benchmark: `SAMPLES = 200`.

In [ ]:
SAMPLES = 50  # quick test; use 200 for full benchmark

!noisekit generate \
  --dataset google/fleurs \
  --config en_us \
  --split test \
  --samples {SAMPLES} \
  --preset clean_reference \
  --preset telecom \
  --preset noise_telecom \
  --output ./data/telephony_speech \
  --seed 42

## 4. Full benchmark (SNAC vs EnCodec)

In [ ]:
!python scripts/run_benchmark.py \
  --data-dir ./data/telephony_speech \
  --device cuda \
  --max-samples {SAMPLES} \
  --pesq \
  --out reports/benchmark.json

## 5. Summary table

In [ ]:
import json
from pathlib import Path

data = json.loads(Path("reports/benchmark.json").read_text())
print(f"{'codec|preset':40} {'STOI':>8} {'PESQ':>8} {'enc_ms':>10} {'dec_ms':>10}")
print("-" * 80)
for key, row in sorted(data["summary"].items()):
    stoi = row.get("stoi_mean")
    pesq = row.get("pesq_mean")
    stoi_s = f"{stoi:8.4f}" if stoi is not None else f"{'n/a':>8}"
    pesq_s = f"{pesq:8.3f}" if pesq is not None else f"{'n/a':>8}"
    enc = row.get("encode_ms_mean") or 0
    dec = row.get("decode_ms_mean") or 0
    print(f"{key:40} {stoi_s} {pesq_s} {enc:10.2f} {dec:10.2f}")

## 6. Download results

In [ ]:
from google.colab import files
from pathlib import Path

for name in ("reports/benchmark.json", "reports/benchmark.csv"):
    p = Path(name)
    if p.exists():
        files.download(str(p))

## 7. (Optional) SNAC coarse-only ablation

In [ ]:
!python scripts/run_snac_ablation.py \
  --data-dir ./data/telephony_speech \
  --device cuda \
  --max-samples {SAMPLES} \
  --out reports/snac_ablation.json